In [1]:
import os
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from cleanlab.filter import find_label_issues

# -----------------------
# Config
# -----------------------
labels_file = "image_labels.txt"
dataset_path = "dataset"        # <-- sửa nếu khác
required_size = (150, 150)      # <-- giữ như bạn đang dùng

# -----------------------
# Load label file
# -----------------------
labels = []
with open(labels_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(": ")
        if len(parts) != 2:
            continue
        fname, lbl = parts[0], int(parts[1])
        labels.append((fname, lbl))

label_map = {fname: lbl for fname, lbl in labels}  # dùng 1 tên: label_map

# -----------------------
# Load images
# -----------------------
X_imgs, y, filenames = [], [], []
for filename in os.listdir(dataset_path):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    lbl = label_map.get(filename)     # FIX: dùng đúng label_map
    if lbl is None:
        continue

    img_path = os.path.join(dataset_path, filename)
    try:
        with Image.open(img_path).convert("RGB") as img:
            img = img.resize(required_size)
            arr = np.array(img, dtype=np.float32) / 255.0  # normalize [0,1]
            X_imgs.append(arr)
            y.append(lbl)
            filenames.append(filename)
    except Exception as e:
        print(f"Error loading {filename}: {e}")

X_imgs = np.array(X_imgs, dtype=np.float32)   # (N, H, W, C)
y_labels = np.array(y, dtype=np.int64)

print("Loaded images:", X_imgs.shape)
print("Loaded labels:", y_labels.shape)

# -----------------------
# IMPORTANT: XGBoost needs 2D features
# Flatten images -> (N, H*W*C)
# (Nếu bạn muốn dùng MobileNetV2 embeddings như notebook ban đầu thì nói mình,
#  mình sẽ sửa theo hướng đó sẽ tốt hơn flatten)
# -----------------------
X_features = X_imgs.reshape(len(X_imgs), -1).astype(np.float32)
print("X_features shape:", X_features.shape)

# -----------------------
# A) Train/test split (kèm filenames)
# -----------------------
X_train, X_test, y_train, y_test, fn_train, fn_test = train_test_split(
    X_features, y_labels, filenames,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

classes = np.unique(y_labels)
n_classes = len(classes)
print("Train:", X_train.shape, "Test:", X_test.shape, "Classes:", n_classes)

# -----------------------
# Helper: build model
# -----------------------
def make_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

# -----------------------
# B) OOF pred_probs bằng Stratified K-Fold (trên TRAIN)
# -----------------------
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

oof_pred_probs = np.zeros((len(y_train), n_classes), dtype=np.float32)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = make_model()
    model.fit(X_tr, y_tr)

    # predict_proba ra shape (len(val_idx), n_classes)
    oof_pred_probs[val_idx] = model.predict_proba(X_val)

    val_pred = model.predict(X_val)
    fold_acc = accuracy_score(y_val, val_pred)
    print(f"Fold {fold}/{k} accuracy: {fold_acc:.4f}")

print("OOF pred_probs shape:", oof_pred_probs.shape)

# -----------------------
# C) Cleanlab trên TRAIN bằng OOF pred_probs
# -----------------------
issues_train_idx = find_label_issues(
    labels=y_train,
    pred_probs=oof_pred_probs,
    return_indices_ranked_by="self_confidence"
)

print(f"⚠️ Số mẫu train nghi sai nhãn: {len(issues_train_idx)}")

top_n = 20
print("Top nghi sai nhãn (index trong TRAIN):", issues_train_idx[:top_n])

# In ra filename tương ứng (rất hữu ích để kiểm tra thủ công)
print("\nTop nghi sai nhãn (filename, label):")
for idx in issues_train_idx[:top_n]:
    print(idx, fn_train[idx], "label=", int(y_train[idx]))

# -----------------------
# D) Train final model trên TRAIN, evaluate trên TEST
# -----------------------
final_model = make_model()
final_model.fit(X_train, y_train)

y_pred_test = final_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)

print("\n✅ Test accuracy:", test_acc)
print("\nClassification report:\n", classification_report(y_test, y_pred_test))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_test))

Loaded images: (8707, 150, 150, 3)
Loaded labels: (8707,)
X_features shape: (8707, 67500)
Train: (6965, 67500) Test: (1742, 67500) Classes: 8


XGBoostError: [21:43:36] C:\actions-runner\_work\xgboost\xgboost\src\common\io.h:362: bad_malloc: Failed to allocate 11094787616 bytes.